In [1]:
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [12]:
import pandas as pd

In [ ]:
df=pd.read_csv('../data/milestone1_syedumairali.csv')
df.head()

,id,sender,subject,body,priority,triage_label,clean_text,keywords,triage,triage_match
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,reminder the client meeting is scheduled at t...,"['reminder', 'client', 'meeting', 'scheduled',...",respond_or_act,\bmeeting\b
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,your invoice of inr is due on please pay to ...,"['invoice', 'inr', 'due', 'please', 'pay', 'av...",notify_human,\binvoice\b
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,reminder the client meeting is scheduled at t...,"['reminder', 'client', 'meeting', 'scheduled',...",respond_or_act,\bmeeting\b
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,hello team please find the attached weekly rep...,"['hello', 'team', 'please', 'find', 'attached'...",respond_or_act,\battached\b
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,hello team please find the attached weekly rep...,"['hello', 'team', 'please', 'find', 'attached'...",respond_or_act,\battached\b


In [14]:
def triage_rule(text):
    text = text.lower()

    if "refund" in text or "urgent" in text or "password" in text:
        return "notify_human"

    if "newsletter" in text or "promotion" in text:
        return "ignore"

    return "respond_or_act"

df['triage'] = df['clean_text'].apply(triage_rule)
df[['body', 'triage']].head()

,body,triage
0,Reminder: The client meeting is scheduled at 1...,respond_or_act
1,Your invoice of INR 25515.09 is due on 2025-12...,respond_or_act
2,Reminder: The client meeting is scheduled at 1...,respond_or_act
3,"Hello team, please find the attached weekly re...",respond_or_act
4,"Hello team, please find the attached weekly re...",respond_or_act


In [15]:
df['predicted_triage'] = df['clean_text'].apply(triage_rule)
df[['body','predicted_triage']].head()

,body,predicted_triage
0,Reminder: The client meeting is scheduled at 1...,respond_or_act
1,Your invoice of INR 25515.09 is due on 2025-12...,respond_or_act
2,Reminder: The client meeting is scheduled at 1...,respond_or_act
3,"Hello team, please find the attached weekly re...",respond_or_act
4,"Hello team, please find the attached weekly re...",respond_or_act


In [16]:
#select 100 test emails
eval_df = df.sample(100,random_state=42)
eval_df = eval_df.reset_index(drop=True)
eval_df.head()

,id,sender,subject,body,priority,triage_label,clean_text,keywords,triage,triage_match,predicted_triage
0,96,no-reply@service.com,Weekly Newsletter,Your order #3634 has been shipped and is expec...,low,ignore,your order has been shipped and is expected t...,"['order', 'shipped', 'expected', 'deliver']",respond_or_act,default_fallback,respond_or_act
1,16,news@techblog.com,Payment Overdue,"Hi, don't miss our sale with discounts up to 7...",low,notify_human,hi dont miss our sale with discounts up to on...,"['hi', 'dont', 'miss', 'sale', 'discounts', 's...",respond_or_act,\bsale\b,respond_or_act
2,31,news@techblog.com,Weekly Newsletter,Notice: Your account will be locked unless ver...,high,notify_human,notice your account will be locked unless veri...,"['notice', 'account', 'locked', 'unless', 'ver...",respond_or_act,default_fallback,respond_or_act
3,159,no-reply@service.com,Welcome to Service,Reminder: The client meeting is scheduled at 9...,low,respond,reminder the client meeting is scheduled at t...,"['reminder', 'client', 'meeting', 'scheduled',...",respond_or_act,\bmeeting\b,respond_or_act
4,129,sales@shop.com,Invoice Due,Your order #6464 has been shipped and is expec...,low,respond,your order has been shipped and is expected t...,"['order', 'shipped', 'expected', 'deliver']",respond_or_act,default_fallback,respond_or_act


In [17]:
# Define ideal response logic
def ideal_response_logic(label):
    if label == 'ignore':
        return 'No action needed'
    elif label == 'respond':
        return 'Send a reply'
    elif label == 'respond_or_act':
        return 'Respond and take required action'
    elif label == 'notify_human':
        return 'Escalate to human'
    else:
        return 'Manual review required'

In [18]:
df['ideal_response'] = df['triage_label'].apply(ideal_response_logic)
df[['triage_label', 'ideal_response']].head()

,triage_label,ideal_response
0,notify_human,Escalate to human
1,respond,Send a reply
2,ignore,No action needed
3,respond,Send a reply
4,respond,Send a reply


In [1]:
import pandas as pd
df= pd.read_csv('../data/email_evaluation_dataset_syedumairali.csv')
df.head()

,email_id,email_text,expected_action,expected_tone
0,E001,Quick reminder about today’s client call. Agen...,notify,neutral
1,E002,"Hi, our meeting has been moved to 3 PM today. ...",notify,polite
2,E003,Calendar invite shared for next Monday’s discu...,ignore,neutral
3,E004,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,neutral
4,E005,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,neutral


In [2]:
# Apply your assistant function to the dataset
def email_assistant(email_text):
    text= email_text.lower()
    if "urgent" in text or "immediate" in text or "by end of day" in text:
        action = "notify"
        tone = "urgent"
    elif "please" in text or "could you" in text:
        action = "respond"
        tone = "polite"
    elif "no action required" in text or "fyi" in text:
        action = "ignore"
        tone = "neutral"
    else:
        action = "respond"
        tone = "neutral"

    return {
        "predicted_action": action,
        "predicted_tone": tone
    }

In [3]:
def run_assistant(row):
    result = email_assistant(row["email_text"])
    return pd.Series(result)

df_predictions = df.apply(run_assistant, axis=1)

In [4]:
# Create separate output columns
df[["predicted_action", "predicted_tone"]] = (
    df["email_text"]
    .apply(email_assistant)
    .apply(pd.Series)
)


In [5]:
df[[
    "email_text",
    "expected_action",
    "expected_tone",
    "predicted_action",
    "predicted_tone"
]].head(10)


,email_text,expected_action,expected_tone,predicted_action,predicted_tone
0,Quick reminder about today’s client call. Agen...,notify,neutral,respond,neutral
1,"Hi, our meeting has been moved to 3 PM today. ...",notify,polite,respond,neutral
2,Calendar invite shared for next Monday’s discu...,ignore,neutral,respond,neutral
3,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,neutral,respond,polite
4,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,neutral,respond,polite
5,Weekly standup scheduled as usual. See you there.,ignore,neutral,respond,neutral
6,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,neutral,respond,polite
7,Quick reminder about today’s client call. Agen...,notify,neutral,respond,neutral
8,Weekly standup scheduled as usual. See you there.,ignore,neutral,respond,neutral
9,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,neutral,respond,polite


In [6]:
# Compare predictions with expected values
df["action_correct"] = (
    df["predicted_action"] == df["expected_action"]
)

df["tone_correct"] = (
    df["predicted_tone"] == df["expected_tone"]
)


In [8]:
df[[
    "email_text",
    "expected_action",
    "predicted_action",
    "action_correct",
    "expected_tone",
    "predicted_tone",
    "tone_correct"
]].head(20)


,email_text,expected_action,predicted_action,action_correct,expected_tone,predicted_tone,tone_correct
0,Quick reminder about today’s client call. Agen...,notify,respond,False,neutral,neutral,True
1,"Hi, our meeting has been moved to 3 PM today. ...",notify,respond,False,polite,neutral,False
2,Calendar invite shared for next Monday’s discu...,ignore,respond,False,neutral,neutral,True
3,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,respond,False,neutral,polite,False
4,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,respond,False,neutral,polite,False
5,Weekly standup scheduled as usual. See you there.,ignore,respond,False,neutral,neutral,True
6,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,respond,False,neutral,polite,False
7,Quick reminder about today’s client call. Agen...,notify,respond,False,neutral,neutral,True
8,Weekly standup scheduled as usual. See you there.,ignore,respond,False,neutral,neutral,True
9,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,respond,False,neutral,polite,False


In [ ]:
# Calculate overall accuracy
action_accuracy = df["action_correct"].mean() * 100
tone_accuracy = df["tone_correct"].mean() * 100
print(f"Action Accuracy: {action_accuracy:.2f}%")
print(f"Tone Accuracy: {tone_accuracy:.2f}%")


Action Accuracy: 41.00%
Tone Accuracy: 66.00%


In [14]:
total_emails = len(df)

print(f"Total Emails Evaluated: {total_emails}")
print(f"Correct Action Predictions: {df['action_correct'].sum()}")
print(f"Correct Tone Predictions: {df['tone_correct'].sum()}")
# show sample counts

Total Emails Evaluated: 100
Correct Action Predictions: 41
Correct Tone Predictions: 66


In [16]:
# Filter failed action predictions
action_errors = df[df["action_correct"] == False]
# Display the relevant columns
action_errors_display = action_errors[[
    "email_text",
    "expected_action",
    "predicted_action"
]]

action_errors_display.head(10)


,email_text,expected_action,predicted_action
0,Quick reminder about today’s client call. Agen...,notify,respond
1,"Hi, our meeting has been moved to 3 PM today. ...",notify,respond
2,Calendar invite shared for next Monday’s discu...,ignore,respond
3,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,respond
4,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,respond
5,Weekly standup scheduled as usual. See you there.,ignore,respond
6,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,respond
7,Quick reminder about today’s client call. Agen...,notify,respond
8,Weekly standup scheduled as usual. See you there.,ignore,respond
9,Reminder: Project sync tomorrow at 10 AM. Plea...,notify,respond


In [20]:
# Count how many failures you have
num_errors = len(action_errors)
total = len(df)

print(f"Action Errors: {num_errors} out of {total}")
print(f"Error Rate: {(num_errors / total) * 100:.2f}%")

# Group errors by expected action
action_errors.groupby("expected_action").size()



Action Errors: 59 out of 100
Error Rate: 59.00%


expected_action
ignore    31
notify    28
dtype: int64

In [ ]:
# Save the final DataFrame
output_path = "../data/milestone2_output_syedumairali.csv"

df.to_csv(output_path, index=False)

print(f"File saved to: {output_path}")

File saved to: ../data/milestone2_output_syedumairali.csv


<!-- 
Which type of emails were hardest to classify?
Things like polite reminders, FYI updates that sound like requests, and academic or invoice emails that are important but don’t clearly demand a reply. These sit between notify and respond and confuse rule-based logic

Why did your rules fail in some cases?
Rules look for keywords, not intent. Words like “please” or “kindly” don’t always mean a reply is required. Context, purpose, and implied urgency are hard to capture with fixed conditions, so some emails were misclassified.

How could an LLM improve this process?
An LLM understands meaning, not just words. It can infer intent, urgency, and tone from the full context of the email, handle ambiguous cases better, and adapt to varied writing styles. This makes decisions closer to how a human would judge the email. 
-->

In [9]:
from langsmith import Client
client = Client()


In [10]:
# define the judge prompt
judge_prompt = """
You are a evaluator. Compare the model output with the ideal answer.
Check :
1. Action correctness
2. Tone correctness

Give score :
1 = correct answer
0 = incorrect answer
"""

In [16]:
# Run agent and judge
def evaluate(agent_output,ideal_action,ideal_tone):
    if agent_output["action"]==ideal_action and agent_output["tone"] == ideal_tone:
        return 1
    else:
        return 0

In [17]:
agent_output = {
    "action" : "notify",
    "tone" : "urgent"
}

In [18]:
agent_output ={"action":"notify", "tone":"urgent"}

In [19]:
ideal_action ="notify"
ideal_tone ="urgent"

In [20]:
score = evaluate(agent_output ,ideal_action,ideal_tone)
score

1

In [1]:
import pandas as pd 
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,urgent
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,polite
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,notify,urgent
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,notify,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,notify,neutral


In [ ]:
# Email Assistant Logic
def email_assistant(email_text):
    text = email_text.lower()
    if "urgent" in text  or "deadline" in text:
        return "notify", "urgent"
    elif "thank you" in text or "thanks" in text:
        return "ignore", "polite"
    else :
        return "respond", "neutral"
    
    

In [ ]:
# run the evaluation on sample data
score = []